# 02 — Stiefel DRGD Optimization

This notebook loads a trained Stiefel score model and runs DRGD optimization with the built-in Brockett objective.

### Setup

Install dependencies and select the execution device.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import torch

# If the repo is not present in /content, set REPO_URL to your GitHub repo and rerun.
REPO_URL = "https://github.com/<your-org>/score-manifold-optimization.git"
REPO_DIR = Path("/content/score-manifold-optimization")

if not (REPO_DIR / "pyproject.toml").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_DIR = REPO_DIR / "outputs/stiefel_train_demo"
DATASET_PATH = REPO_DIR / "data/stiefel_n3_p3.pt"
OUTPUT_DIR = REPO_DIR / "outputs/stiefel_opt_demo"

print(f"Repo: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"CHECKPOINT_DIR={CHECKPOINT_DIR}")
print(f"DATASET_PATH={DATASET_PATH}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

### Local Setup (Alternative)

If the repo is already cloned locally, run this cell instead of the Colab setup cell above.

In [ ]:
from pathlib import Path
import os
import torch

# Local alternative: use this when the repository is already cloned.
start = Path.cwd().resolve()
REPO_DIR = next((p for p in [start, *start.parents] if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    raise RuntimeError("Could not find repo root (missing pyproject.toml in parent dirs).")

os.chdir(REPO_DIR)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_DIR = REPO_DIR / "outputs/stiefel_train_demo"
DATASET_PATH = REPO_DIR / "data/stiefel_n3_p3.pt"
OUTPUT_DIR = REPO_DIR / "outputs/stiefel_opt_demo"

print(f"Repo root: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"CHECKPOINT_DIR={CHECKPOINT_DIR}")
print(f"DATASET_PATH={DATASET_PATH}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

# Install once from terminal (repo root): pip install -e .


### Run Optimization

Execute DRGD optimization with the Brockett objective.

In [ ]:
import shutil

import torch

from diffusion.optimization import (
    RiemannianConfig,
    ScoreTangentConfig,
    build_score_projector,
    run_riemannian_optimization,
)
from diffusion.utils.checkpoint_utils import load_pretrained_score_context

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(42)
context = load_pretrained_score_context(
    checkpoint_dir=CHECKPOINT_DIR,
    dataset_path_override=DATASET_PATH,
    device=DEVICE,
)
constraint = context.constraint
train_data = context.train_data
test_data = context.test_data

projector = build_score_projector(
    score_model=context.model,
    diffusion=context.diffusion,
    score_time=0.25,
    data_dims=train_data.shape[1:],
    n_proj=1,
    tangent_cfg=ScoreTangentConfig(method="jvp"),
)

# Custom quadratic Brockett objective example: f(X) = tr(X^T Q X K)
n, p = train_data.shape[-2], train_data.shape[-1]
Q = torch.randn((n, n), device=train_data.device, dtype=train_data.dtype)
Q = 0.5 * (Q + Q.transpose(-1, -2))
K = torch.diag(torch.randn((p,), device=train_data.device, dtype=train_data.dtype))

def objective_fn(x):
    xtqx = x.transpose(1, 2) @ Q @ x
    return torch.einsum("bij,ji->b", xtqx, K)

#Initial initial point (minimum of train data)
min_idx = objective_fn(train_data).argmin()
x0 = train_data[min_idx].unsqueeze(0).detach().clone().to(torch.device(DEVICE))
initial_objective = float(objective_fn(x0).detach().item())

#Optimization config
opt_cfg = RiemannianConfig(
        step_size=2e-2,
        n_steps=30,
        tangent_mode="projector",
    )

result = run_riemannian_optimization(
    x0=x0,
    objective_fn=objective_fn,
    grad_objective_fn=None,
    projector=projector,
    cfg=opt_cfg,
)

history_objective = [float(objective_fn(x).mean().detach().item()) for x in result.trajectory]
history_objective_projected = [float(objective_fn(constraint.project(x)).mean().detach().item()) for x in result.trajectory]
history_distance = [float(constraint.distance(x).mean().detach().item()) for x in result.trajectory]


### Plot Optimization Diagnostics

Plot objective values and manifold distance over optimization iterations.

In [ ]:
import matplotlib.pyplot as plt

iters = range(len(history_objective))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

#Compute the minimum on train data
min_train_objective = float(objective_fn(train_data).min().detach().item())

#Compute the theoretical minimum on for the given quadratic cost function
eig_q = torch.sort(torch.linalg.eigvalsh(Q), descending=False).values
diag_k = torch.sort(torch.diagonal(K), descending=True).values
theoretical_min = float(torch.dot(eig_q, diag_k).detach().item())

axes[0].plot(iters, history_objective, label="Iterates Objective")
axes[0].plot(iters, history_objective_projected, color="orange", label="Projected Iterates Objective")
axes[0].axhline(min_train_objective, color="green", linestyle="--", label="Minimum on training data")
axes[0].axhline(theoretical_min, color="red", linestyle="--", label="Theoretical minimum")
axes[0].set_title("Objective vs Iteration")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Objective")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(iters, history_distance)
axes[1].set_title("Manifold Distance vs Iteration")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Mean Distance")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

initial_objective = history_objective[0]
final_objective = history_objective[-1]
delta = final_objective - initial_objective
traj_len = len(result.trajectory)
final_x_shape = tuple(result.final_x.shape)

print(f"Initial objective: {initial_objective:.6f}")
print(f"Final objective: {final_objective:.6f}")
print(f"Objective delta (final - initial): {delta:.6f}")
print(f"Objective reduction (initial - final): {initial_objective - final_objective:.6f}")
print(f"Trajectory length: {traj_len}")
print(f"Final tensor shape: {final_x_shape}")